In [ ]:
import os
import subprocess
import sys
import glob
import re
import shutil

# NLGCL Kaggle Runner v15 (Fixed: Match QB-Video official preprocessing - Tab Separator & No Neg Sampling)
# 1. Clone the repository if not present
if not os.path.exists('main.py'):
    !git clone https://github.com/yangzeha/NLGCL.git
    %cd NLGCL

# 2. Patch Code for Numpy 2.0 Compatibility 
print("Patching RecBole for Numpy 2.x compatibility...")
files = glob.glob('recbole/**/*.py', recursive=True) + glob.glob('recbole_gnn/**/*.py', recursive=True)

replacements = [
    (r'np\.float\b', 'float'),
    (r'np\.int\b', 'int'),
    (r'np\.object\b', 'object'),
    (r'np\.bool\b', 'bool'),
    (r'np\.str\b', 'str'),
    (r'np\.long\b', 'int'),
]

patched_count = 0
for file_path in files:
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        new_content = content
        
        # Disable generic compatibility_settings to avoid crashes
        if 'def compatibility_settings(self):' in new_content:
            new_content = re.sub(
                r'def compatibility_settings\(self\):.*?(?=\n\s+def|\Z)', 
                'def compatibility_settings(self):\n        pass\n\n    ', 
                new_content, 
                flags=re.DOTALL
            )

        # Safe replacements for types
        for pattern, replacement in replacements:
            new_content = re.sub(pattern, replacement, new_content)
            
        if new_content != content:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(new_content)
            patched_count += 1
    except Exception as e:
        print(f"Skipping {file_path}: {e}")

print(f"Successfully patched {patched_count} files.")

# 3. Setup Dependencies
print("Configuring environment...")
!pip install "numpy>=2.0.0" "pandas>=2.2.2" --force-reinstall --upgrade
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.4.0+cu121.html

requirements_content = """colorlog
tensorboard
six
colorama
pyyaml
hyperopt>=0.2.7
torch_geometric
"""
with open('requirements.txt', 'w') as f:
    f.write(requirements_content)
!pip install -r requirements.txt

# 4. Dataset Preprocessing (Matches NLGCL Paper & Tenrec Official)
preprocess_script = r"""
import os
import pandas as pd
import re
import shutil

# --------------------------
# Config: Match NLGCL Paper + Tenrec Official
# --------------------------
DATASET_DIR = 'dataset/QB-video'
INTER_FILE = os.path.join(DATASET_DIR, 'QB-video.inter')
RAW_CSV_PATH = 'dataset/QB-video.csv'
K_CORE = 5  # Paper requires k=5
FIELD_SEPARATOR = '\t'  # Tenrec official separator is tab

# Create dataset directory
os.makedirs(DATASET_DIR, exist_ok=True)

# --------------------------
# Step 1: Load Raw Data
# --------------------------
source_path = None
if os.path.exists(RAW_CSV_PATH):
    source_path = RAW_CSV_PATH
elif os.path.exists(INTER_FILE):
    # Fallback if raw csv missing but inter exists (though we prefer raw for fresh start)
    source_path = INTER_FILE

if not source_path:
    print("Error: QB-video.csv not found.")
    exit(1)

print(f"Loading raw data from: {source_path}")

# Note: The input CSV might be comma-separated even if the target is Tab.
# We auto-detect for robustness.
try:
    df = pd.read_csv(source_path) 
    if len(df.columns) < 2: 
        # If read incorrectly, try tab
        df = pd.read_csv(source_path, sep='\t')
except:
    df = pd.read_csv(source_path, sep='\t')

print(f"Loaded columns: {list(df.columns)}")

# --------------------------
# Step 2: Keep Core Columns Only
# --------------------------
if 'video_id' in df.columns:
    df.rename(columns={'video_id': 'item_id'}, inplace=True)

required_cols = ['user_id', 'item_id']
if not all(col in df.columns for col in required_cols):
    print(f"Error: Missing core columns. Found: {df.columns.tolist()}")
    exit(1)

# Keep ONLY interactions (NLGCL uses pure interaction matrix)
df = df[required_cols].copy()

# --------------------------
# Step 3: Deduplication
# --------------------------
original_len = len(df)
df = df.drop_duplicates(subset=['user_id', 'item_id'], keep='first')
print(f"Deduplication: {original_len} -> {len(df)}")

# --------------------------
# Step 4: Recursive k-core Filtering (Strict Match)
# --------------------------
def apply_k_core(df, k=5, user_col='user_id', item_col='item_id'):
    print(f"Starting {k}-core filtering...")
    print(f"Initial: Interactions={len(df)}, Users={df[user_col].nunique()}, Items={df[item_col].nunique()}")
    
    iteration = 0
    while True:
        iteration += 1
        n_rows_before = len(df)
        
        # Filter Users (count >= k)
        user_counts = df[user_col].value_counts()
        valid_users = user_counts[user_counts >= k].index
        df = df[df[user_col].isin(valid_users)]
        
        # Filter Items (count >= k)
        item_counts = df[item_col].value_counts()
        valid_items = item_counts[item_counts >= k].index
        df = df[df[item_col].isin(valid_items)]
        
        n_rows_after = len(df)
        # print(f"  Iter {iteration}: {n_rows_after} remaining")
        
        if n_rows_after == n_rows_before:
            break
            
    print(f"Finished: Interactions={len(df)}, Users={df[user_col].nunique()}, Items={df[item_col].nunique()}")
    return df

df = apply_k_core(df, k=K_CORE)

# --------------------------
# Step 5: Save as RecBole format
# --------------------------
# Add token remapping suffix
df.columns = ['user_id:token', 'item_id:token']
# Save using TAB separator as per official spec
df.to_csv(INTER_FILE, index=False, sep=FIELD_SEPARATOR)
print(f"Saved preprocessed data to: {INTER_FILE}")

# --------------------------
# Step 6: Fix Configuration
# --------------------------
config_path = 'properties/QB-video.yaml'
if os.path.exists(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    new_content = content
    
    # 1. Start fresh with field_separator
    # Remove old field_separator lines if any
    new_content = re.sub(r'field_separator:.*', '', new_content)
    # Ensure standard load_col is correct
    if 'inter: [user_id, item_id]' in new_content:
        new_content = new_content.replace('inter: [user_id, item_id]', 'inter: [user_id, item_id]\nfield_separator: "\\t"')
    else:
        new_content += '\nfield_separator: "\\t"'

    # 2. Add k-core intervals (Redundant safety net)
    if 'user_inter_num_interval' not in new_content:
        new_content += f'\nuser_inter_num_interval: "[{K_CORE},inf)"'
    if 'item_inter_num_interval' not in new_content:
        new_content += f'\nitem_inter_num_interval: "[{K_CORE},inf)"'

    # 3. Clean up bad keys
    if 'val_interval' in new_content:
        new_content = re.sub(r'val_interval:.*', '', new_content)
    
    # 4. Inject NLGCL Paper Hyperparameters for QB-Video
    # RecSys 2025 values provided by user
    params_to_inject = [
        "n_layers: 4",
        "reg_weight: 0.0001",
        "cl_temp: 0.2",
        "alpha: 0.6",
        "cl_reg: 5e-5"
    ]
    
    for param in params_to_inject:
        key = param.split(':')[0]
        if key not in new_content:
            new_content += f"\n{param}"
        else:
            # Update existing
            new_content = re.sub(f'{key}:.*', param, new_content)

    with open(config_path, 'w', encoding='utf-8') as f:
        f.write(new_content)
    print("Updated properties/QB-video.yaml with paper parameters.")

# --------------------------
# Step 7: Clear Caches
# --------------------------
cache_dirs = ['dataset_cache', 'saved']
for d in cache_dirs:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Cleared cache directory: {d}")
"""

with open('preprocess_dataset.py', 'w', encoding='utf-8') as f:
    f.write(preprocess_script)

print("Running preprocessing in isolated subprocess...")
!python preprocess_dataset.py

# 5. Start Training
print("Starting Training...")
!python main.py --dataset QB-video